# 09 -- Melon rush: all units grow MELON exclusively

Motivated by a real finding in the episode 96612755 replay analysis
(`PROGRESS.md`, 2026-08-22): the winning opponent's melon sales alone
were $25,594 -- **81%** of their $31,477 final money, from 220 units
sold starting day 10 (`MELON`'s `first_yield_day`, a hard floor for
everyone -- the market is shared between both players, so nobody can
sell melon before that turn, us or them). `wheat_loop_melon_agent`'s
single 4-tile hand is far too small to compete at that scale (45
`PLANT MELON` orders for the opponent vs. our 3).

This experiment tests the other extreme: **every unit** (farmer + up to
5 hands) grows MELON from turn 0, maximizing production/volume. No
wheat at all -- deliberately, to isolate the raw effect before adding
any phase-switching logic. Real risk to check via `evaluate.py` rather
than assume: no income until day 10, against seed ($80/unit) and hire
costs from the $3000 starting money.

In [1]:
import sys
from pathlib import Path

from kaggle_environments import make

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kaggriculture_bot.evaluate import evaluate
from kaggriculture_bot.strategies.baseline import (
    DIRECTIONS,
    HAND_ZONES,
    LOOP_ORDER_12,
    LOOP_ORDER_HANDS,
    SPAWN_FEEDER,
    wheat_loop_melon_agent,
    melon_rush_agent,
)

## Design

Reuses `wheat_loop_hands_agent`'s exact structure (`LOOP_ORDER_12` for
the farmer, `HAND_ZONES` for hands) -- only `"WHEAT"` -> `"MELON"` and
harvest threshold `2` -> `10` (melon's `first_yield_day`), applied to
every unit, not just one hand. `melon_rush_agent` already graduated to
`src/kaggriculture_bot/strategies/baseline.py` (imported above) --
mechanical enough (same structure as an already-validated agent, just a
uniform crop/threshold swap) that it didn't need the notebook's usual
cell-by-cell build step. This notebook's job is validating it.

## Validate against the current best

Current best is `wheat_loop_melon_agent` (4-tile melon hand + wheat
elsewhere). Run twice, per this project's rule.

In [2]:
print(evaluate(wheat_loop_melon_agent, melon_rush_agent, n=20))

{'win_rate': 0.0, 'avg_money': 16219.0}


In [3]:
print(evaluate(wheat_loop_melon_agent, melon_rush_agent, n=20))

{'win_rate': 0.0, 'avg_money': 15216.0}


## If it loses: check whether it went broke

Quick diagnostic -- log money over the first ~15 days of one episode to
see if the no-income-until-day-10 risk actually bites.

In [4]:
diag_env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
diag_env.run([melon_rush_agent, "random"])

for step_idx in range(0, 360, 24):
    obs = diag_env.steps[step_idx][0]["observation"]
    farm = obs["farms"][obs["player"]]
    print("day", obs["day"], "money:", farm["money"], "hands:", len(farm["hands"]))

day 0 money: 3000.0 hands: 0
day 1 money: 828.0 hands: 0
day 2 money: 576.0 hands: 0
day 3 money: 484.0 hands: 0
day 4 money: 472.0 hands: 0
day 5 money: 460.0 hands: 0
day 6 money: 448.0 hands: 0
day 7 money: 436.0 hands: 0
day 8 money: 424.0 hands: 0
day 9 money: 412.0 hands: 0
day 10 money: 400.0 hands: 0
day 11 money: 68.0 hands: 0
day 12 money: 5266.0 hands: 0
day 13 money: 11235.0 hands: 0
day 14 money: 17068.0 hands: 0
